In [1]:
import pandas as pd 
import numpy as np

# Setting Up

In [2]:
movies = pd.read_csv('/Users/samreenasiddiqui/Desktop/cinematch_recommender/data/data_1m/movies.dat', sep='::', engine='python', encoding='latin-1',
    names=['movie_id', 'title', 'genres'])

movies.head()

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings = pd.read_csv('/Users/samreenasiddiqui/Desktop/cinematch_recommender/data/data_1m/ratings.dat', sep='::',engine='python', encoding='latin-1',
                      names = ["user_id","movie_id","rating","timestamp"])
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'],unit='s')

ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,2000-12-31 22:12:40
1,1,661,3,2000-12-31 22:35:09
2,1,914,3,2000-12-31 22:32:48
3,1,3408,4,2000-12-31 22:04:35
4,1,2355,5,2001-01-06 23:38:11


In [4]:
users = pd.read_csv('/Users/samreenasiddiqui/Desktop/cinematch_recommender/data/data_1m/users.dat', sep='::', engine='python', encoding='latin-1',
             names = ["user_id","gender","age","occupation","zip-code"]        )

users.head()

,user_id,gender,age,occupation,zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [5]:
full_table = ratings.merge(right = movies , on = ['movie_id'], how = 'left')
full_table = full_table.merge(right = users, on = ['user_id'], how = 'left')
full_table[['user_id','gender','age', 'occupation','zip-code','movie_id','title','genres', 'rating','timestamp']]
full_table

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip-code
0,1,1193,5,2000-12-31 22:12:40,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067
1,1,661,3,2000-12-31 22:35:09,James and the Giant Peach (1996),Animation|Children's|Musical,F,1,10,48067
2,1,914,3,2000-12-31 22:32:48,My Fair Lady (1964),Musical|Romance,F,1,10,48067
3,1,3408,4,2000-12-31 22:04:35,Erin Brockovich (2000),Drama,F,1,10,48067
4,1,2355,5,2001-01-06 23:38:11,"Bug's Life, A (1998)",Animation|Children's|Comedy,F,1,10,48067
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,2000-04-26 02:35:41,Weekend at Bernie's (1989),Comedy,M,25,6,11106
1000205,6040,1094,5,2000-04-25 23:21:27,"Crying Game, The (1992)",Drama|Romance|War,M,25,6,11106
1000206,6040,562,5,2000-04-25 23:19:06,Welcome to the Dollhouse (1995),Comedy|Drama,M,25,6,11106
1000207,6040,1096,4,2000-04-26 02:20:48,Sophie's Choice (1982),Drama,M,25,6,11106


In [6]:
full_table['relevant'] = full_table['rating'] >= 4
full_table

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip-code,relevant
0,1,1193,5,2000-12-31 22:12:40,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067,True
1,1,661,3,2000-12-31 22:35:09,James and the Giant Peach (1996),Animation|Children's|Musical,F,1,10,48067,False
2,1,914,3,2000-12-31 22:32:48,My Fair Lady (1964),Musical|Romance,F,1,10,48067,False
3,1,3408,4,2000-12-31 22:04:35,Erin Brockovich (2000),Drama,F,1,10,48067,True
4,1,2355,5,2001-01-06 23:38:11,"Bug's Life, A (1998)",Animation|Children's|Comedy,F,1,10,48067,True
...,...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,2000-04-26 02:35:41,Weekend at Bernie's (1989),Comedy,M,25,6,11106,False
1000205,6040,1094,5,2000-04-25 23:21:27,"Crying Game, The (1992)",Drama|Romance|War,M,25,6,11106,True
1000206,6040,562,5,2000-04-25 23:19:06,Welcome to the Dollhouse (1995),Comedy|Drama,M,25,6,11106,True
1000207,6040,1096,4,2000-04-26 02:20:48,Sophie's Choice (1982),Drama,M,25,6,11106,True


# Exploration

In [5]:
len(users)

6040

In [6]:
len(movies)

3883

In [7]:
len(ratings)

1000209

In [8]:
avg_amt_ratings = ratings.groupby("user_id")['rating'].agg(['count', 'mean']).reset_index()
avg_amt_ratings

,user_id,count,mean
0,1,53,4.188679
1,2,129,3.713178
2,3,51,3.901961
3,4,21,4.190476
4,5,198,3.146465
...,...,...,...
6035,6036,888,3.302928
6036,6037,202,3.717822
6037,6038,20,3.800000
6038,6039,123,3.878049


In [9]:
avg_amt_ratings.sort_values(by = 'count', ascending = False)

,user_id,count,mean
4168,4169,2314,3.551858
1679,1680,1850,3.555676
4276,4277,1743,4.134825
1940,1941,1595,3.054545
1180,1181,1521,2.815911
...,...,...,...
5724,5725,20,3.450000
3406,3407,20,3.500000
1663,1664,20,3.400000
4418,4419,20,3.300000


In [10]:
avg_movie_rantings = ratings.groupby("movie_id")['rating'].agg(['count', 'mean']).reset_index()
avg_movie_rantings.sort_values(by = 'count', ascending = False)

,movie_id,count,mean
2651,2858,3428,4.317386
253,260,2991,4.453694
1106,1196,2990,4.292977
1120,1210,2883,4.022893
466,480,2672,3.763847
...,...,...,...
3013,3237,1,1.000000
725,763,1,3.000000
607,624,1,4.000000
2367,2563,1,3.000000


In [12]:
full_table['rating'].describe()

count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

In [13]:
full_table.groupby('genres')['rating'].describe().sort_values(by='mean', ascending=False)

,count,mean,std,min,25%,50%,75%,max
genres,,,,,,,,
Animation|Comedy|Thriller,688.0,4.473837,0.739339,1.0,4.0,5.0,5.00,5.0
Sci-Fi|War,1367.0,4.449890,0.805507,1.0,4.0,5.0,5.00,5.0
Animation,459.0,4.394336,0.819555,1.0,4.0,5.0,5.00,5.0
Film-Noir|Mystery,1584.0,4.367424,0.776372,1.0,4.0,5.0,5.00,5.0
Adventure|War,1644.0,4.346107,0.794099,1.0,4.0,5.0,5.00,5.0
...,...,...,...,...,...,...,...,...
Action|Adventure|Children's|Fantasy,44.0,2.090909,1.074399,1.0,1.0,2.0,3.00,5.0
Comedy|Film-Noir|Thriller,5.0,2.000000,1.000000,1.0,1.0,2.0,3.00,3.0
Action|Adventure|Children's|Sci-Fi,350.0,1.874286,1.044111,1.0,1.0,2.0,2.00,5.0


In [15]:
p_relevant = sum(full_table['relevant'])/ len(full_table)

In [16]:
full_table.groupby('user_id')['relevant'].mean().sort_values().reset_index()

,user_id,relevant
0,4486,0.000000
1,3598,0.000000
2,5850,0.017241
3,2744,0.072464
4,4349,0.074074
...,...,...
6035,447,1.000000
6036,283,1.000000
6037,4755,1.000000
6038,2543,1.000000


In [17]:
full_table.groupby('movie_id')['relevant'].mean().sort_values().reset_index()

,movie_id,relevant
0,2576,0.0
1,847,0.0
2,843,0.0
3,2592,0.0
4,3458,0.0
...,...,...
3701,1316,1.0
3702,2480,1.0
3703,3353,1.0
3704,439,1.0


# Split and General Popularity Ranking 

In [ ]:
import sys 
sys.path.append("..")
from src.recommender.data.split import chron_train_val_test
train, val, test= chron_train_val_test(full_table)

In [8]:
popular_movies = full_table.groupby("movie_id").size().sort_values(ascending = False)
popular_movies.head(10)

movie_id
2858    3428
260     2991
1196    2990
1210    2883
480     2672
2028    2653
589     2649
2571    2590
1270    2583
593     2578
dtype: int64

In [9]:
#general recommendations 

user_id = 1
seen_movies = set(train[train['user_id'] == user_id]['movie_id'])

reccomended_movies = [movie for movie in popular_movies.index if movie not in seen_movies][:10] #top popular movies they haven't seen (train = past)
reccomended_movies

movies[movies["movie_id"].isin(reccomended_movies)][['movie_id','title']]

relevant_test_movies = set(test.loc[(test['user_id'] == user_id) & (test['relevant']), "movie_id"]) #movies they saw in the future 
relevant_test_movies

hits = set(reccomended_movies) & set(relevant_test_movies) #how many "recomended" movies did they end up seeing 
hits

precision_at_10 = len(hits)/10
precision_at_10
recall_at_10 = len(hits)/len(relevant_test_movies)
recall_at_10

reccomended_movies

[2858, 1196, 1210, 480, 589, 2571, 593, 1580, 1198, 110]

In [32]:
from src.recommender.models.popularity import PopularityRecommender
popularity_model = PopularityRecommender()
popularity_model.fit(train)
recommendations = popularity_model.recommend(user_id = 1, k=10)
recommendations


[2858, 1196, 1198, 593, 2571, 1210, 589, 318, 858, 110]

In [11]:
from src.recommender.evaluation.metrics import precision_at_k, recall_at_k, ndcg_at_k
#test 

recommended = [1,2,3,4,5]
relevant = [2,4,8]

print(precision_at_k(recommended, relevant, k=5))
print(recall_at_k(recommended, relevant, k=5))

0.4
0.6666666666666666


In [ ]:
# test. model is trained on what they have seen (train_df). now if we give it unseen val data, how do the recommendations look? are there matches with
# what they actually watched (val)?
results = []
relevant_val = val[val["relevant"]]

for user_id, user_data in relevant_val.groupby("user_id"):
    relevant_movies = set(user_data["movie_id"])

    recomendations = popularity_model.recommend(user_id, k = 10)

    precision = precision_at_k(recomendations, relevant_movies, k=10)
    recall = recall_at_k(recommendations, relevant_movies, k=10)
    ndcg = ndcg_at_k(recommendations, relevant_movies, k=10)

    results.append({
            "user_id": user_id,
            "precision": precision, 
            "recall": recall,
            "ndcg": ndcg
    }
    )

results_df = pd.DataFrame(results)

results_df
results_df[["precision","recall","ndcg"]].mean()


precision    0.044610
recall       0.025138
ndcg         0.024321
dtype: float64

In [13]:
recommended = [1,2,3,4,5]
relevant = [1,2]

print(ndcg_at_k(recommended, relevant, k = 5))

1.0


In [ ]:
from src.recommender.evaluation.evaluator import evaluate_model

results_df = evaluate_model(popularity_model, val, k=10)
results_df.mean(numeric_only=True)

user_id            3034.768832
precision_at_10       0.044610
recall_at_10          0.046939
ndcg_at_10            0.056883
dtype: float64

# Content Based Recommender

In [33]:
from src.recommender.models.content import ContentRecommender

content_model = ContentRecommender()
content_model.fit(train, movies)
recs = content_model.recommend(user_id = 1 , k = 10, return_scores = True)
recs
# rec_movies = movies[movies['movie_id'].isin(recs)]
# rec_movies


[(np.int64(34), 0.8016449488040421),
 (np.int64(1014), 0.8016449488040421),
 (np.int64(1812), 0.8016449488040421),
 (np.int64(917), 0.8009507034596427),
 (np.int64(1013), 0.8009507034596427),
 (np.int64(2059), 0.8009507034596427),
 (np.int64(1547), 0.8009507034596427),
 (np.int64(1026), 0.8009507034596427),
 (np.int64(1012), 0.8009507034596427),
 (np.int64(262), 0.8009507034596427)]

In [17]:
results_df_content = evaluate_model(content_model, val, k=10)
results_df_content.mean(numeric_only=True)

user_id            3034.768832
precision_at_10       0.007602
recall_at_10          0.014761
ndcg_at_10            0.011525
dtype: float64

# Item CF (colab finding)

In [34]:
from src.recommender.models.item_cf import ItemCFRecommender

cf_model = ItemCFRecommender()
cf_model.fit(train)
recs = cf_model.recommend(user_id = 1 , k = 10, return_scores = True)
recs
# rec_movies = movies[movies['movie_id'].isin(recs)]
# rec_movies

[(np.int64(1196), 11.879802841540803),
 (np.int64(1198), 11.652040327089978),
 (np.int64(318), 11.625526838996649),
 (np.int64(593), 11.205076473983711),
 (np.int64(1), 10.936067670344341),
 (np.int64(296), 10.572603653233204),
 (np.int64(2858), 10.471900059423458),
 (np.int64(858), 10.442197940136946),
 (np.int64(1259), 10.431414685160028),
 (np.int64(1210), 10.327855352479531)]

In [11]:
from src.recommender.evaluation.evaluator import evaluate_model
results_df_cf = evaluate_model(cf_model, val, k=10)
results_df_cf.mean(numeric_only=True)

user_id            3034.768832
precision_at_10       0.060332
recall_at_10          0.080247
ndcg_at_10            0.085332
dtype: float64

# ALS

In [35]:
from src.recommender.models.als import ALSRecommender

als_model = ALSRecommender()
als_model.fit(train)
recs = als_model.recommend(user_id = 1 , k = 10, return_scores = True)
recs
# rec_movies = movies[movies['movie_id'].isin(recs)]
# rec_movies


  0%|          | 0/20 [00:00<?, ?it/s]

[(np.int64(318), 0.5623466372489929),
 (np.int64(593), 0.4673624634742737),
 (np.int64(364), 0.35445478558540344),
 (np.int64(1259), 0.35069572925567627),
 (np.int64(1196), 0.3429606258869171),
 (np.int64(3471), 0.34012705087661743),
 (np.int64(1), 0.3369714021682739),
 (np.int64(34), 0.3218501806259155),
 (np.int64(1225), 0.31415197253227234),
 (np.int64(588), 0.31051281094551086)]

In [23]:
from src.recommender.evaluation.evaluator import evaluate_model
results_df_als = evaluate_model(als_model, val, k=10)
results_df_als.mean(numeric_only=True)

user_id            3034.768832
precision_at_10       0.061508
recall_at_10          0.097656
ndcg_at_10            0.090455
dtype: float64

In [14]:
import mlflow
mlflow.set_experiment('als-tuning')

2026/08/17 21:10:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/17 21:10:02 INFO mlflow.store.db.utils: Updating database tables
2026/08/17 21:10:02 INFO mlflow.tracking.fluent: Experiment with name 'als-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/samreenasiddiqui/Desktop/cinematch_recommender/notebooks/mlruns/1', creation_time=1787026202955, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787026202955, lifecycle_stage='active', name='als-tuning', tags={}, trace_location=None, workspace='default'>

In [15]:
from src.recommender.models.als import ALSRecommender
from src.recommender.evaluation.evaluator import evaluate_model
configs = [
    {'factors': 32, "regularization": 0.01, "iterations": 20},
    {'factors': 64, "regularization": 0.01, "iterations": 20},
    {'factors': 128, "regularization": 0.01, "iterations": 20},

    {'factors': 64, "regularization": 0.05, "iterations": 20},
    {'factors': 64, "regularization": 0.10, "iterations": 20},

    {'factors': 64, "regularization": 0.01, "iterations": 20},
]


experiment_results = []

for config in configs: 

    with mlflow.start_run():
        mlflow.log_params(config)
    
        model = ALSRecommender( 
            factors = config['factors'], 
            regularization = config['regularization'], 
            iterations = config['iterations']
        )

        model.fit(train)
        results = evaluate_model(model, val, k=10)

        metrics = results[['precision_at_10', 'recall_at_10','ndcg_at_10']].mean()

        mlflow.log_metrics({ 
            "precision": metrics['precision_at_10'],
            'recall': metrics['recall_at_10'],
            'ndcg': metrics['ndcg_at_10']
        })


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [17]:
import mlflow 

print(mlflow.get_tracking_uri())

sqlite:////Users/samreenasiddiqui/Desktop/cinematch_recommender/notebooks/mlflow.db


In [8]:
from src.recommender.models.als import ALSRecommender
als_model = ALSRecommender()
als_model.fit(train)
als_model.recommend(1, 10)

  0%|          | 0/20 [00:00<?, ?it/s]

[np.int64(318),
 np.int64(593),
 np.int64(364),
 np.int64(1259),
 np.int64(1196),
 np.int64(3471),
 np.int64(1),
 np.int64(34),
 np.int64(1225),
 np.int64(588)]

In [9]:
als_model.recommend(1,10,True)

[(np.int64(318), 0.5623466372489929),
 (np.int64(593), 0.4673624634742737),
 (np.int64(364), 0.35445478558540344),
 (np.int64(1259), 0.35069572925567627),
 (np.int64(1196), 0.3429606258869171),
 (np.int64(3471), 0.34012705087661743),
 (np.int64(1), 0.3369714021682739),
 (np.int64(34), 0.3218501806259155),
 (np.int64(1225), 0.31415197253227234),
 (np.int64(588), 0.31051281094551086)]

# Candidate Builder 

In [36]:
from src.recommender.ranking.candidates import build_candidate_table

models = {
    "popularity" : popularity_model, 
    "content": content_model,
    "item_cf" : cf_model, 
    "als" : als_model
}

candidate_df = build_candidate_table(models, val, 50)
candidate_df.head()

,user_id,movie_id,popularity_score,popularity_rank,item_cf_score,item_cf_rank,als_score,als_rank,label,content_score,content_rank
0,1,2858,2648.0,1.0,10.471900,7.0,0.209059,29.0,0,NaN,NaN
1,1,1196,2311.0,2.0,11.879803,1.0,0.342961,5.0,0,NaN,NaN
2,1,1198,2065.0,3.0,11.652040,2.0,0.275649,16.0,0,NaN,NaN
3,1,593,2016.0,4.0,11.205076,4.0,0.467362,2.0,0,NaN,NaN
4,1,2571,1973.0,5.0,10.031536,20.0,NaN,NaN,0,NaN,NaN


In [13]:
candidate_recall_by_user = []
relevant_val = val[val['relevant']].groupby('user_id')['movie_id'].apply(set)
candidates_by_user = candidate_df.groupby("user_id")['movie_id'].apply(set)

for user_id, relevant_movies in relevant_val.items():
    candidate_movies = candidates_by_user.get(user_id, set())
    hits = relevant_movies & candidate_movies 
    recall = len(hits) / len(relevant_movies)
    candidate_recall_by_user.append(recall)

candidate_recall = sum(candidate_recall_by_user) / len(candidate_recall_by_user)

candidate_recall

0.4246543540219343

In [14]:
candidate_df_100 = build_candidate_table(models, val, 100)

candidate_recall_by_user_100 = []
relevant_val = val[val['relevant']].groupby('user_id')['movie_id'].apply(set)
candidates_by_user_100 = candidate_df_100.groupby("user_id")['movie_id'].apply(set)

for user_id, relevant_movies in relevant_val.items():
    candidate_movies_100 = candidates_by_user_100.get(user_id, set())
    hits = relevant_movies & candidate_movies_100 
    recall = len(hits) / len(relevant_movies)
    candidate_recall_by_user_100.append(recall)

candidate_recall_100 = sum(candidate_recall_by_user_100) / len(candidate_recall_by_user_100)

candidate_recall_100

0.5872533907419247

In [15]:
candidate_df_100

,user_id,movie_id,popularity_score,popularity_rank,item_cf_score,item_cf_rank,als_score,als_rank,label,content_score,content_rank
0,1,2858,2648.0,1.0,10.471900,7.0,0.209059,29.0,0,NaN,NaN
1,1,1196,2311.0,2.0,11.879803,1.0,0.342961,5.0,0,NaN,NaN
2,1,1198,2065.0,3.0,11.652040,2.0,0.275649,16.0,0,NaN,NaN
3,1,593,2016.0,4.0,11.205076,4.0,0.467362,2.0,0,NaN,NaN
4,1,2571,1973.0,5.0,10.031536,20.0,0.116566,95.0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1547087,6040,2732,NaN,NaN,NaN,NaN,0.202917,93.0,0,NaN,NaN
1547088,6040,1885,NaN,NaN,NaN,NaN,0.202733,94.0,0,NaN,NaN
1547089,6040,1249,NaN,NaN,NaN,NaN,0.201920,95.0,1,NaN,NaN
1547090,6040,2919,NaN,NaN,NaN,NaN,0.200360,98.0,1,NaN,NaN


In [16]:
print(candidate_df_100['label'].value_counts())

print(candidate_df_100['label'].value_counts(normalize = True))

label
0    1519745
1      27347
Name: count, dtype: int64
label
0    0.982324
1    0.017676
Name: proportion, dtype: float64


# XGBoost Preparation 

In [17]:
from sklearn.model_selection import train_test_split 

user_ids = candidate_df_100['user_id'].unique()
rank_train_users, rank_val_users = train_test_split(user_ids, test_size = 0.2, random_state = 42)

rank_train_df = candidate_df_100[candidate_df_100['user_id'].isin(rank_train_users)].copy()
rank_train_df = rank_train_df.sort_values("user_id")
rank_val_df = candidate_df_100[candidate_df_100['user_id'].isin(rank_val_users)].copy()
rank_val_df = rank_val_df.sort_values("user_id")

print(rank_train_df['user_id'].nunique())
print(rank_val_df['user_id'].nunique())

print(
    len(
        set(rank_train_df['user_id']) & set(rank_val_df['user_id'])
    )
)

4630
1158
0


In [18]:
feature_cols = ["popularity_score", "popularity_rank","content_score","content_rank","item_cf_score","item_cf_rank","als_score","als_rank"]

X_train = rank_train_df[feature_cols] #removing movie id and user id from training. 
y_train = rank_train_df['label']
qid_train = rank_train_df['user_id'] #dont want to train on user_ids for patterns. 
                                        #needs this so xgboost knows which candidate movies should be compared against one another 

X_val = rank_val_df[feature_cols]
y_val = rank_val_df['label']
qid_val = rank_val_df['user_id']

In [19]:
from xgboost import XGBRanker

ranker = XGBRanker(
    objective = "rank:ndcg", #should one particular movie be ranked above another (not is movie A relevant, or predict movie A)
    n_estimators = 300, 
    learning_rate = 0.05, 
    max_depth = 6, 
    subsample = 0.8, 
    colsample_bytree = 0.8, 
    tree_method = 'hist',
    random_state = 42
)

ranker.fit(X_train, y_train, qid = qid_train, 
           eval_set = [(X_val, y_val)],
           eval_qid = [qid_val],
           verbose = False)


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [22]:
from src.recommender.evaluation.metrics import precision_at_k, recall_at_k, ndcg_at_k
rank_val_df = rank_val_df.copy()
rank_val_df['ranker_score'] = ranker.predict(rank_val_df[feature_cols])

ranker_results = []
for user_id, user_candidates in rank_val_df.groupby("user_id"):
    top_10 = (user_candidates.sort_values('ranker_score', ascending = False).head(10)['movie_id'].tolist())

    relevant_movies = set(val.loc[(val['user_id'] == user_id) & (val['relevant']), "movie_id"])

    ranker_results.append({
        "user_id": user_id,
        "precision": precision_at_k(top_10, relevant_movies, 10), 
        "recall": recall_at_k(top_10, relevant_movies, 10), 
        "ndcg": ndcg_at_k(top_10, relevant_movies, 10)
    })

ranker_results_df = pd.DataFrame(ranker_results)
ranker_results_df[['precision','recall','ndcg']].mean()

precision    0.059413
recall       0.094547
ndcg         0.091578
dtype: float64

In [25]:
print(results_df_als[["precision_at_10",'recall_at_10','ndcg_at_10']].mean().round(5))

precision_at_10    0.06151
recall_at_10       0.09766
ndcg_at_10         0.09045
dtype: float64


In [27]:
als_comparison_df = val[val['user_id'].isin(rank_val_users)]

als_same_users = evaluate_model(als_model, als_comparison_df, k = 10)
als_same_users[['precision_at_10', 'recall_at_10','ndcg_at_10']].mean()

precision_at_10    0.057599
recall_at_10       0.095102
ndcg_at_10         0.086695
dtype: float64

In [28]:
import mlflow 

mlflow.set_experiment("hybrid-maker")

with mlflow.start_run():

    mlflow.log_params({
        "objective": "rank:ndcg",
        "n_estimators": 300, 
        "learning_rate": 0.05, 
        "max_depth": 6, 
        "subsample": 0.8, 
        "colsample_bytree": 0.8, 
        "candidate_k": 100, 
        "num_rank_train_users": 4630, 
        "num_rank_val_user": 1158
    })

    mlflow.log_metrics({
        "precision_at_10": 0.059413,
        "recall_at_10": 0.094547,
        "ndcg_at_10": 0.091578
    })

2026/08/18 14:57:09 INFO mlflow.tracking.fluent: Experiment with name 'hybrid-maker' does not exist. Creating a new experiment.


# ReRunning All Including Val Set

In [29]:
candidate_df_100 = candidate_df_100.sort_values("user_id")

X_final = candidate_df_100[feature_cols]
y_final = candidate_df_100['label']
qid_final = candidate_df_100['user_id']

final_ranker = XGBRanker(
    objective = 'rank:ndcg',
    n_estimators = 300, 
    learning_rate = 0.05, 
    max_depth = 6, 
    subsample = 0.8, 
    colsample_bytree = 0.8, 
    tree_method = 'hist',
    random_state = 42
)

final_ranker.fit(X_final, y_final, qid = qid_final, verbose = False)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [30]:
final_train_df = pd.concat([train, val], ignore_index = True)

In [37]:
final_popularity = PopularityRecommender()
final_popularity.fit(final_train_df)

final_content = ContentRecommender()
final_content.fit(final_train_df, movies)

final_item_cf = ItemCFRecommender()
final_item_cf.fit(final_train_df)

final_als = ALSRecommender(factors = 32, regularization=0.01, iterations = 20)
final_als.fit(final_train_df)

  0%|          | 0/20 [00:00<?, ?it/s]

In [38]:
final_models = {
    'popularity': final_popularity,
    'content': final_content, 
    'item_cf': final_item_cf, 
    'als': final_als
}

test_candidates = build_candidate_table(models = final_models, evaluation_df = test, candidate_k= 100)
test_candidates['ranker_score'] = final_ranker.predict(test_candidates[feature_cols])

test_results = []

for user_id, user_candidates in test_candidates.groupby("user_id"):
    top_10 = user_candidates.sort_values('ranker_score', ascending = False).head(10)['movie_id'].tolist()
    relevant_movies = set(test.loc[(test['user_id'] == user_id) & (test['relevant']), 'movie_id'])

    test_results.append({
        'user_id': user_id,
        'precision': precision_at_k(top_10, relevant_movies, 10),
        'recall': recall_at_k(top_10, relevant_movies, 10), 
        'ndcg': ndcg_at_k(top_10,relevant_movies, 10)
    })

test_results_df = pd.DataFrame(test_results)

test_results_df[['precision','recall','ndcg']].mean()

precision    0.059223
recall       0.090564
ndcg         0.084631
dtype: float64